# v13 仮説検証 ― 順方向×逆方向の一致度が高い波形ほど真に近い?

保存済みPF波形(`fwd_traj`(32,T) / `bwd_traj`(128,T))だけで検証(PF・U-Net不要)。

- ペア: 逆方向128 = 順方向32 × 各4。`bwd→(32,4,T)`、`fwd[i]`↔`bwd[i,j]` の 128ペア。
- 各ペアの一致度 = 坑井全体の `rmse(fwd[i], bwd[i,j])`。**最小K(=5)ペア**を採用。
- 予測TVT(坑井ごと, 列ごと):
  - **A 順方向**: 採用ペアの `fwd[i]` 平均
  - **B 順+逆平均**: 採用ペアの `(fwd[i]+bwd[i,j])/2` 平均
  - **A,Bの平均**
- ベースライン: **順方向32本の単純平均**(+参考: center=尤度加重平均)
- 各案の **overall RMSE**(全坑井プール)を比較。

真値 = `center + target_rel`。RMSEは絶対TVTで評価(=Kaggle整合)。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
# img/(生波形)ディレクトリを自動検出(パスが2通りあるため)
CAND = [PROJECT/"v13"/"artifacts_v13"/"img", PROJECT/"artifacts_v13"/"img"]
RAW_DIR = next((d for d in CAND if d.exists()), None)
assert RAW_DIR is not None, f"img/ が見つからない: {CAND}"
print("RAW_DIR =", RAW_DIR)

FWD = 32; BWD_PER = 4          # 1順方向あたり逆方向の本数
K = 5                          # 採用する最一致ペア数
wells = sorted(p.stem for p in RAW_DIR.glob("*.npz"))
print("wells:", len(wells))

RAW_DIR = C:\Users\kosaka256\Documents\rogii_claude\v13\artifacts_v13\img
wells: 773


In [2]:
def rmse_pool(errs):
    e = np.concatenate(errs); e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e**2)))

def well_preds(wid, K):
    d = np.load(RAW_DIR / f"{wid}.npz")
    if "target_rel" not in d.files:
        return None
    center = d["center"].astype(np.float64)
    true = center + d["target_rel"].astype(np.float64)        # 絶対TVT 真値
    fwd = d["fwd_traj"].astype(np.float64)                    # (32,T)
    bwd = d["bwd_traj"].astype(np.float64)                    # (128,T)
    T = len(center)
    bwd = bwd.reshape(FWD, BWD_PER, T)                        # (32,4,T) bwd[i,j]=fwd[i]の子

    # 各ペアの一致度(坑井全体RMSE) → 小さいほど一致
    diff = fwd[:, None, :] - bwd                              # (32,4,T)
    pair_rmse = np.sqrt(np.nanmean(diff**2, axis=2))          # (32,4)
    flat = pair_rmse.ravel()
    sel = np.argsort(flat)[:K]                                # 最一致K
    si, sj = np.unravel_index(sel, (FWD, BWD_PER))

    base = fwd.mean(0)                                        # 順方向32本の単純平均
    A = fwd[si].mean(0)                                       # 採用ペアの順方向 平均
    B = ((fwd[si] + bwd[si, sj]) * 0.5).mean(0)              # 採用ペアの順+逆平均
    AB = (A + B) * 0.5
    return dict(true=true, base=base, A=A, B=B, AB=AB, center=center)

errs = {k: [] for k in ["base", "A", "B", "AB", "center"]}
nwell = 0
for w in wells:
    r = well_preds(w, K)
    if r is None: continue
    nwell += 1
    for k in errs:
        errs[k].append(r[k] - r["true"])
print(f"集計坑井: {nwell}")

集計坑井: 773


In [3]:
names = {
    "base":   "順方向32本 単純平均(ベースライン)",
    "center": "順方向 尤度加重平均(=pf_ancc, 参考)",
    "A":      f"一致上位{K}: 順方向 平均",
    "B":      f"一致上位{K}: 順+逆 平均",
    "AB":     f"一致上位{K}: A,Bの平均",
}
base = rmse_pool(errs["base"])
print(f"{'手法':40s}{'overall RMSE':>14}{'Δ vs base':>12}")
for k in ["base", "center", "A", "B", "AB"]:
    r = rmse_pool(errs[k])
    print(f"{names[k]:40s}{r:>14.4f}{r-base:>12.4f}")
print("\nΔ<0 なら順方向単純平均より改善。仮説(一致度高い=真に近い)が効くか確認。")

手法                                        overall RMSE   Δ vs base
順方向32本 単純平均(ベースライン)                             9.8778      0.0000
順方向 尤度加重平均(=pf_ancc, 参考)                        9.1877     -0.6901
一致上位5: 順方向 平均                                  10.4295      0.5516
一致上位5: 順+逆 平均                                  10.2022      0.3243
一致上位5: A,Bの平均                                  10.2305      0.3526

Δ<0 なら順方向単純平均より改善。仮説(一致度高い=真に近い)が効くか確認。


In [4]:
# K を振って傾向を見る(任意)
print(f"{'K':>4}{'A(順)':>12}{'B(順+逆)':>12}{'AB':>12}   (base=%.4f)" % base)
for Kk in [1, 3, 5, 10, 20]:
    e = {"A": [], "B": [], "AB": []}
    for w in wells:
        r = well_preds(w, Kk)
        if r is None: continue
        for k in e: e[k].append(r[k] - r["true"])
    print(f"{Kk:>4}{rmse_pool(e['A']):>12.4f}{rmse_pool(e['B']):>12.4f}{rmse_pool(e['AB']):>12.4f}")

   K        A(順)      B(順+逆)          AB   (base=9.8778)
   1     11.7050     11.4771     11.4924
   3     10.6733     10.4339     10.4681
   5     10.4295     10.2022     10.2305
  10     10.0930      9.8494      9.8797
  20      9.8531      9.5821      9.6142
